In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
import pickle 


In [ ]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

In [ ]:
##data = data.drop(['RowNumber','Surname','CustomerId'], ais = 1)

In [ ]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [ ]:
data.head()

In [ ]:
ohe_encoder_geo = OneHotEncoder()
geo_encoded = ohe_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = ohe_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

In [ ]:
data = data.drop(['Geography'], axis=1)
data = pd.concat([data, geo_encoded_df], axis=1)
data

In [ ]:
##Dividing into inputs and outputs
X = data.drop(['EstimatedSalary'], axis=1)
Y = data['EstimatedSalary']

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)



In [ ]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(ohe_encoder_geo, file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)

ANN Regression Problem

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
##creating the model
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_absolute_error', metrics = ['mae'])

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights= True)

In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_data = (X_test, Y_test),
    epochs = 100, 
    callbacks = [early_stopping_callback, tensorboard_callback]
)

In [ ]:
test_loss, test_mae = model.evaluate(X_test, Y_test)
print(f'Mean Absolute Error : {test_mae}')

In [ ]:
model.save('regression_model.h5')